# Lab 3.1: Data Pre-Processing

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates the data preprocessing workflow for extracting and preparing Sentinel-2 satellite imagery and CORINE land cover maps for land classification tasks.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 1 | HPC Access Setup | ✅ Previous |
| Lab 2 | Jupyter-JSC & Git | ✅ Previous |
| Lab 3.1 | **Data Preprocessing** | 🔄 **Current** |
| Lab 3.2 | Google Earth Engine Acquisition | ⬜ Next |
| Lab 4 | Understanding Transformers | ⬜ Next |
| Lab 4.1 | Training on Sentinel-2 Data | ⬜ Next |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## What You'll Learn

By the end of this lab, you will:
- Read and inspect geospatial raster files using GDAL
- Understand coordinate reference systems (CRS) and reprojection
- Extract CORINE land cover maps to match Sentinel-2 tiles
- Process Sentinel-2 SAFE format archives into GeoTIFF
- Batch process multiple tiles on HPC systems

## Quick Start

This lab focuses on the **Data Acquisition & Preprocessing** phase of the project pipeline:

```
Sentinel-2 Download (Lab 3.2) 
    ↓
Data Preprocessing (Lab 3.1) ← You are here
    ↓
CORINE Label Extraction (Lab 3.1)
    ↓
Training Data Preparation (Lab 4)
    ↓
Model Training (Lab 4.1, 5)
    ↓
Validation & Evaluation (Lab 6)
    ↓
Foundation Models (Lab 7)
```

---

## Overview

The preprocessing pipeline consists of:
1. **Reading Raster Data**: Opening and inspecting geospatial raster files
2. **Extracting CORINE Land Cover Maps**: Reprojecting and extracting CORINE data to match Sentinel-2 tiles
3. **Processing Sentinel-2 Data**: Extracting and converting Sentinel-2 SAFE format to GeoTIFF
4. **HPC Batch Processing**: Submitting jobs to process multiple tiles on HPC systems

## Part 1: Opening and Inspecting Raster Data

First, we'll learn how to read and inspect raster files using GDAL.

In [ ]:
from osgeo import gdal
import os
import numpy as np

### Opening a Raster File

In [ ]:
# Open raster file
rasterPath = "path_to_tif_folder/s2_tile_ID.tif"

ds = gdal.Open(rasterPath, 0)
band_ds = ds.GetRasterBand(1)
data = band_ds.ReadAsArray()
print(data)

### Inspecting Data Shape

In [ ]:
print(data.shape)

In [ ]:
print(data[2000:2002, 2000:2002])

### Printing Raster Metadata

In [ ]:
# Get metadata as string
metadata = gdal.Info(ds)
print(metadata)

### Getting JSON Metadata and WGS84 Extent

In [ ]:
# Get metadata in JSON format
import json
info = json.loads(gdal.Info(ds, format='json'))
print(info)

In [ ]:
# Extract WGS84 extent
wgs84Extent = info['wgs84Extent']
print(wgs84Extent)

## Part 2: Extracting CORINE Land Cover Map

Now we'll extract and reproject the CORINE land cover map to match our Sentinel-2 tile's coordinate system and extent.

### Checking Coordinate Reference Systems

In [ ]:
# Check CRS of Sentinel-2 image
s2ImgPath = "path_to_tif_folder/s2_tile_ID.tif"
s2 = gdal.Open(s2ImgPath, 0)
crs = s2.GetProjection()
print(crs)

### Reprojecting CORINE Map

Reproject the CORINE map to match the Sentinel-2 CRS using `gdal.Warp`.

In [ ]:
# Reproject CORINE map
corineMapPath = "path_to_corine_folder/U2018_CLC2018_V2020_20u1.tif"
output_path = "path_to_corine_folder/corine_tile_ID_reproj.tif"

gdal.Warp(output_path, corineMapPath, dstSRS=crs)

### Extracting Tile Coordinates

In [ ]:
# Get coordinates from Sentinel-2 tile
ulx, xres, xskew, uly, yskew, yres = s2.GetGeoTransform()
lrx = ulx + (s2.RasterXSize * xres)
lry = uly + (s2.RasterYSize * yres)

print(f"Upper Left: ({ulx}, {uly})")
print(f"Lower Right: ({lrx}, {lry})")

### Extracting CORINE Map for Tile

Use `gdal_translate` to extract the CORINE map extent matching the Sentinel-2 tile.

In [ ]:
# Extract CORINE map using tile coordinates
input_reproj_path = "path_to_corine_folder/corine_tile_ID_reproj.tif"
output_extract_path = "path_to_corine_folder/corine_tile_ID.tif"

os.system(f'gdal_translate -projwin {ulx} {uly} {lrx} {lry} {input_reproj_path} {output_extract_path}')

## Part 3: Sentinel-2 Data Extraction Script

This section contains the Python functions for extracting Sentinel-2 data from SAFE format and converting it to GeoTIFF.

In [ ]:
from osgeo import gdal
import numpy as np
import os
from datetime import datetime as dt
from zipfile import ZipFile
import argparse

### Function: Create GeoTIFF

Creates a GeoTIFF file from a numpy array with proper georeferencing.

In [ ]:
def creategeotiff(name, array, NDV, dataSorce, dataType):
    """
    Create a GeoTIFF file from a numpy array.
    
    Parameters:
    -----------
    name : str
        Output filename
    array : numpy.ndarray
        3D array (height, width, bands)
    NDV : numeric
        No Data Value
    dataSorce : gdal.Dataset
        Source dataset for georeferencing
    dataType : int
        GDAL data type (e.g., gdal.GDT_UInt16)
    
    Returns:
    --------
    str : Output filename
    """
    driver = gdal.GetDriverByName('GTiff')
    array[np.isnan(array)] = NDV
    dataSet = driver.Create(name, array.shape[1], array.shape[0], array.shape[2], dataType)
    dataSet.SetGeoTransform(dataSorce.GetGeoTransform())
    dataSet.SetProjection(dataSorce.GetProjection()) 

    for i in range(0, array.shape[2]):
        dataSet.GetRasterBand(i+1).WriteArray(array[:, :, i])
    dataSet.FlushCache()
    
    return name

### Function: Read Sentinel-2 SAFE Format

Reads specified bands from a Sentinel-2 SAFE archive and returns a stacked numpy array.

In [ ]:
def readS2safe(in_zip, bands, tile):
    """
    Read Sentinel-2 SAFE format and extract specified bands.
    
    Parameters:
    -----------
    in_zip : str
        Path to extracted SAFE directory
    bands : list
        List of band names (e.g., ['B02', 'B03', 'B04', 'B08'])
    tile : str
        Tile ID (e.g., '28VCG')
    
    Returns:
    --------
    tuple : (output_img, dataSorce)
        - output_img: numpy array (10980, 10980, n_bands)
        - dataSorce: gdal.Dataset for georeferencing
    """
    flist = []  
    
    for root, dirs, files in os.walk(in_zip):
        for file in files:
            flist.append(os.path.join(root, file))
    
    output_img = np.zeros((10980, 10980, len(bands)))
    dataSorce = None
    
    for i in range(len(bands)):
        band = bands[i]
        
        # Determine resolution
        if band in ['B02', 'B03', 'B04', 'B08']:
            res = 10 
        elif band in ['B05', 'B06', 'B07', 'B8A', 'B11', 'B12', 'SCL']:
            res = 20
        elif band in ['B01', 'B09']:
            res = 60 
        else:
            raise ValueError(f'Bad band values: {band}')
    
        chn_fn = None
        
        # Find band file
        for fname in flist:
            if (band in fname and 'IMG_DATA' in fname and 
                fname.endswith('.jp2') and f'{res}m' in fname and 
                f'_T{tile}' in fname):
                chn_fn = fname
                break
        
        if chn_fn is None:
            raise ValueError(f'Cannot find channel name in zip file: {band}, {res}, tile={tile}')
        
        # Read band
        ds = gdal.Open(chn_fn, 0)
        band_ds = ds.GetRasterBand(1)
        data = (band_ds.ReadAsArray()).astype(np.uint16)
        
        if band == 'B02':
            dataSorce = ds
        
        # Apply percentile clipping
        band_flat = np.reshape(data, (10980 * 10980, 1))
        maxVal = np.quantile(band_flat, 0.999)
        minVal = np.quantile(band_flat, 0.001)
        band_flat[band_flat > maxVal] = maxVal
        band_flat[band_flat < minVal] = minVal
        data = np.reshape(band_flat, (10980, 10980))
        output_img[:, :, i] = data
                
    return output_img, dataSorce

### Function: Unzip and Convert Sentinel-2 Data

Main function to process a SAFE archive and save as GeoTIFF.

In [ ]:
def unzipS2andSaveToTif(safe_path, tif_path, tile):
    """
    Extract Sentinel-2 bands and save as GeoTIFF.
    
    Parameters:
    -----------
    safe_path : str
        Path to .SAFE directory
    tif_path : str
        Output directory for GeoTIFF
    tile : str
        Tile ID
    """
    bands = ['B02', 'B03', 'B04', 'B08']
    output_img, dataSorce = readS2safe(safe_path, bands, tile)
    name = safe_path[:-5]  # Remove .SAFE extension
    output_path = os.path.join(tif_path, name + '.tif')
    creategeotiff(output_path, output_img, 0, dataSorce, gdal.GDT_UInt16)

### Main Processing Script

This section processes all Sentinel-2 zip files in a directory.

In [ ]:
# Example usage (uncomment and modify paths as needed)
# input_path = "path_to_zip_files"
# output_path = "path_to_converted_tif"
# tile = "your_tile_ID"  # e.g., "28VCG"

# ziplist = os.listdir(input_path)
# print("ziplist", ziplist)

# for x in ziplist:
#     if x.endswith(".zip"):
#         name = x.split(".")[0]
#         zip_path = os.path.join(input_path, name + ".zip")
#         print("Processing:", zip_path)
#         
#         with ZipFile(zip_path) as zipObj:
#             zipObj.extractall(output_path)
#         
#         safe_path = os.path.join(output_path, name + ".SAFE")
#         unzipS2andSaveToTif(safe_path, output_path, tile)
# 
# print("Process finished")

## Part 4: HPC Batch Processing with Slurm

For processing large datasets, we can submit the extraction script as a batch job to an HPC cluster using Slurm.

### Slurm Submission Script

Below is the bash script for submitting the Sentinel-2 extraction job to a Slurm-managed HPC system.

In [ ]:
%%bash
# Save this as submit_extract_s2.sh and submit with: sbatch submit_extract_s2.sh

#!/usr/bin/env bash
# Slurm job configuration
#SBATCH --nodes=1
#SBATCH --cpus-per-task=128
#SBATCH --ntasks-per-node=1
#SBATCH --output=extract_s2.out 
#SBATCH --error=extract_s2.err
#SBATCH --time=2:00:00
#SBATCH --job-name=extract_s2
#SBATCH --account=training2328
#SBATCH --partition=dc-cpu

module --force purge
module use $OTHERSTAGES 
ml Stages/2020  
ml GCC/10.3.0  ParaStationMPI/5.4.10-1
ml GDAL/3.1.2-Python-3.8.5
# load virtual environment if needed

##### Number of total processes
echo " "
echo " Nodelist       := " $SLURM_JOB_NODELIST
echo " Number of nodes:= " $SLURM_JOB_NUM_NODES
echo " Ntasks per node:= " $SLURM_NTASKS_PER_NODE
echo " Ntasks         := " $SLURM_NTASKS
echo " "

echo ""
echo "Run started at:- "
date
srun --cpu-bind=none python -u extract_s2.py $input_path "path_to_zip_files" $output_path "path_to_converted_tif" $tile "your_tile_ID"
echo "Run finished at:- "
date

### Submitting the Job

To submit the job to the cluster, save the script above as `submit_extract_s2.sh` and run:

```bash
sbatch submit_extract_s2.sh
```

Monitor job status with:
```bash
squeue -u $USER
```

Check output in `extract_s2.out` and errors in `extract_s2.err`.

## Summary

This notebook covered the complete preprocessing pipeline for Sentinel-2 and CORINE data:

1. **Reading Rasters**: Opening and inspecting GeoTIFF files with GDAL
2. **CORINE Extraction**: Reprojecting and extracting CORINE land cover maps to match S2 tiles
3. **S2 Processing**: Converting Sentinel-2 SAFE archives to GeoTIFF with band stacking
4. **HPC Processing**: Batch processing multiple tiles using Slurm on HPC systems

The output GeoTIFF files are ready for use in land cover classification models.

---

## What's Next?

### Before Moving to Lab 4

**1. Complete Data Preparation**
   - Ensure you have both Sentinel-2 and CORINE data processed for your region
   - Verify all GeoTIFF files are georeferenced correctly

**2. Create Training Dataset**
   - Use the processed files to create training patches
   - Pair Sentinel-2 patches with CORINE labels
   - Save as CSV or HDF5 for efficient loading

**3. Data Splits**
   - Create train/validation/test splits (70/15/15 or similar)
   - Ensure balanced class distribution

### Next Lab: Lab 4 - Understanding Transformers

In **Lab 4**, you'll learn:
- PyTorch fundamentals and tensor operations
- Self-attention mechanisms
- Transformer architecture for remote sensing classification

This is essential background before training your first model in **Lab 4.1**.

### Data Pipeline Recap

```python
# Lab 3.1 produces:
Sentinel-2 GeoTIFF files (10980 x 10980 pixels)
CORINE GeoTIFF files (labeled land cover)
    ↓
# Lab 4 processes:
Extract 3x3 or 5x5 patches
Create training/validation datasets
    ↓
# Lab 4.1 trains:
Transformer model on patches
Save trained weights
```

---

## Resources & References

- **GDAL Documentation**: https://gdal.org/
- **Sentinel-2 Product Specification**: https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2
- **CORINE Land Cover**: https://www.eea.europa.eu/publications/COR0-landcover
- **GeoTIFF Format**: https://www.ogc.org/standards/geotiff

---

## Troubleshooting & FAQ

**Q: How do I handle large Sentinel-2 tiles?**
- Use gdal.SetConfigOption('GDAL_CACHEMAX', 1024) to increase memory cache
- Process tiles in chunks if needed
- Store intermediate results

**Q: What if coordinate systems don't match?**
- Always check with GetProjection() before extraction
- Use gdal.Warp() to reproject, not gdal_translate
- Verify extent with GetGeoTransform()

**Q: Can I process multiple tiles in parallel?**
- Yes! Use the Slurm script with array jobs: `#SBATCH --array=0-100%10`
- Modify the Python script to process one tile per array task

---

**Course Contact**: Refer to course materials for instructor email and office hours
**Last Updated**: January 2026